### Links and Resources
| Concept | Reference |
|---|---|
| [Structured Streaming guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) | The whole streaming model |
| [Built-in functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html) | Same catalog as batch |
| [Watermarks](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#handling-late-data-and-watermarking) | Bounding state |
| [`current_timestamp`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.current_timestamp.html) | Determinism caveat |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# Functions in Streaming

The `pyspark.sql.functions` catalog you've used in batch is **the same catalog** in Structured Streaming. Most functions work identically. A small set don't, and the difference is always about one of two things:

1. **Determinism** — the function returns a different value at different times (e.g. `current_timestamp`, `rand`).
2. **State** — the function needs to see the whole table to answer (e.g. `count_distinct` without watermark).

Notebook 03 in the Streaming folder covers what's *legal* on streams generally. This notebook is specifically about which functions need extra thought.

## A demo source

We'll use a `rate` stream — a synthetic source that emits `value, timestamp` rows at a fixed rate. Easy to reason about, no external setup.

In [ ]:
import time

events = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 20)
    .option("numPartitions", 1)
    .load()
    .withColumnRenamed("value", "event_id")
    .withColumn("user_id", (F.col("event_id") % 5).cast("int")))

events.printSchema()
print("isStreaming:", events.isStreaming)

In [ ]:
def peek(stream_df, name, seconds=4, limit=10):
    q = (stream_df.writeStream
         .format("memory").queryName(name)
         .outputMode("append").start())
    time.sleep(seconds)
    q.stop()
    return spark.sql(f"SELECT * FROM {name} LIMIT {limit}")

## Stateless functions — work exactly like batch

`select`, `withColumn`, `filter`, `concat`, `from_json`, `when`, `regexp_replace`, `length`, `cast` — all the projection / row-wise stuff just works. If the batch version is correct, the streaming version is correct.

In [ ]:
stateless = (events
    .withColumn("bucket",
        F.when(F.col("event_id") % 3 == 0, "premium").otherwise("free"))
    .withColumn("user_label", F.format_string("user_%02d", "user_id"))
    .filter(F.col("user_id") < 3))

display(peek(stateless, "stateless_demo"))

## Time functions — `current_timestamp` and `current_date`

These are **non-deterministic**: each micro-batch sees a different "now". That's usually exactly what you want for a streaming pipeline ("when did we ingest this row?"), but it has consequences:

- The same row replayed after a failure could get a different `current_timestamp`. If your pipeline relies on `current_timestamp` for joins or deduplication, it's broken on retry.
- For event-time logic, **always use the column from the source** (here: `timestamp`), not `current_timestamp`. The source column is what travels with the row through the system.

In [ ]:
timed = events.select(
    "event_id",
    F.col("timestamp").alias("event_time"),     # event time (deterministic)
    F.current_timestamp().alias("ingest_time"), # ingest time (NOT deterministic on retry)
)
display(peek(timed, "timed_demo"))

## Random functions — same warning, harder

`rand()` and `randn()` without a seed change every batch. **With a seed**, they still produce different values for the same row across replays in some Spark versions, because the seed is per-task. If you need a stable random per row, hash a stable column instead:

In [ ]:
stable = events.withColumn(
    "stable_hash",
    F.crc32(F.col("event_id").cast("string")) % 100,   # 0..99, deterministic per event_id
)
display(peek(stable, "stable_demo"))

## Aggregations — fine WITH a watermark, rejected WITHOUT

A streaming aggregation needs to remember partial results until they're "done". An unbounded aggregate (`groupBy(key).count()` over all of time) requires unbounded state — never finishes, eventually OOMs.

A **watermark** tells Spark "you'll never see data older than this", which lets it close out and emit old groups. Once the watermark passes, the state is dropped.

```python
agg = (events
    .withWatermark("timestamp", "10 seconds")
    .groupBy(F.window("timestamp", "10 seconds"), "user_id")
    .agg(F.count("*").alias("n")))
```

We won't run an aggregation here (output mode complicates the demo) — see Streaming notebook 04 for the full treatment. The point: **`groupBy` is allowed; you must watermark first.**

## `count_distinct` and `dropDuplicates`

- **Exact `count_distinct`** on a stream is rejected — the state is unbounded.
- **`approx_count_distinct`** works (HyperLogLog state is bounded).
- **`dropDuplicates(["id"])`** also requires a watermark, because Spark needs to remember which IDs it has seen, and the watermark bounds how far back that memory goes.

In [ ]:
dedup_demo = (events
    .withWatermark("timestamp", "20 seconds")
    .dropDuplicates(["event_id"]))
print("legal because of withWatermark — would be rejected without it")
print("isStreaming:", dedup_demo.isStreaming)

## UDFs in streaming — same flavors, same costs

All three UDF flavors (Python, Arrow, Pandas) work on streaming DataFrames. The performance trade-offs from notebook 15 carry over identically: prefer built-ins, then Pandas UDFs, then Arrow, then classic Python.

One extra rule: **your UDF should be deterministic and side-effect-free.** A retry will replay rows; if your UDF writes to an external system or returns different values for the same input, you'll see double-writes or non-reproducible output.

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("string")
def label(uid: pd.Series) -> pd.Series:
    return uid.map({0: "alice", 1: "bob", 2: "carol", 3: "dave", 4: "eve"}).fillna("unknown")

labeled = events.withColumn("user_name", label("user_id")).select("event_id", "user_id", "user_name")
display(peek(labeled, "labeled_demo"))

## Higher-order functions on arrays — fully supported

Stateless and per-row, so streaming works exactly like batch.

In [ ]:
with_array = events.withColumn(
    "tags",
    F.array(
        F.lit("new"),
        F.when(F.col("event_id") % 2 == 0, F.lit("even")).otherwise(F.lit("odd")),
    ),
)

transformed = with_array.select(
    "event_id",
    F.transform("tags", lambda t: F.upper(t)).alias("upper_tags"),
    F.exists("tags", lambda t: t == F.lit("even")).alias("has_even"),
)
display(peek(transformed, "hof_demo"))

## Quick rules-of-thumb table

| Function category | Streaming-safe? | Notes |
|---|---|---|
| Projection / row-wise built-ins (`when`, `concat`, `from_json`, regex…) | Yes | Identical to batch |
| Higher-order array functions (`transform`, `filter`, `aggregate`) | Yes | Per-row, stateless |
| Window over partition (`row_number`, `lag`) | **No** | Needs all rows — use stream-stream join + state instead |
| `groupBy().agg(...)` | Yes, with watermark | One streaming aggregation per query |
| `count_distinct` (exact) | **No** | Use `approx_count_distinct` |
| `dropDuplicates` | Yes, with watermark | Without watermark = unbounded state |
| `current_timestamp` / `current_date` | Yes, but non-deterministic | Use event-time column for logic |
| `rand()` / `randn()` | Yes, but non-deterministic | Use a hash of a stable column for repeatability |
| Python / Arrow / Pandas UDFs | Yes | Must be deterministic & side-effect-free |

## Takeaways

- The `pyspark.sql.functions` catalog is the same in streaming. Most things just work.
- **Stateless = always fine.** Stateful = needs a **watermark** to bound state.
- Use the source's event-time column for time-based logic, not `current_timestamp`. Replays must be reproducible.
- `count_distinct` exact is forbidden; `approx_count_distinct` is your friend.
- UDFs work but should be deterministic — retries replay rows.